# PennyLane QFT and phase estimation

Estimate a representable phase with three counting wires and compare the probability distribution.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the quantum problem

QPE combines controlled phase accumulation with an inverse QFT to produce a phase distribution.

In [2]:
counting = 3
phase = 3 / 8

def make_qnode(device):
    @qml.qnode(device)
    def circuit():
        qml.PauliX(counting)
        for wire in range(counting):
            qml.Hadamard(wire)
            qml.ControlledPhaseShift(2 * np.pi * phase * (2 ** wire), wires=[wire, counting])
        qml.adjoint(qml.QFT)(wires=range(counting))
        return qml.probs(wires=range(counting))
    return circuit

reference_qnode = make_qnode(qml.device("default.qubit", wires=counting + 1))

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(reference_qnode)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
mettleq_device = MettleQDevice(wires=counting + 1, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(mettleq_qnode)
error = max_abs_error(reference, candidate)
mode_match = int(np.argmax(reference)) == int(np.argmax(candidate))
method, device = pennylane_selection(mettleq_device)

## 4. Check correctness before discussing speed

The distributions and most likely phase bin must match.

In [5]:
tutorial_result = emit_result(
    notebook="pennylane/07_qft_and_qpe.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="phase probabilities atol=3e-6 and identical mode",
    passed=error <= 3e-6 and mode_match,
    exact_match=mode_match,
    selected_method=method,
    selected_device=device,
    metrics={"max_probability_error": error, "expected_phase": phase, "mode": int(np.argmax(candidate))},
)


Comparison summary
------------------
Correctness contract: PASS — phase probabilities atol=3e-6 and identical mode
SDK reference median: 1.271 ms
MettleQ median:       1.407 ms
Timing interpretation: the SDK reference was 1.107x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: yes

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "phase probabilities atol=3e-6 and identical mode", "exact_match": true, "framework": "pennylane", "machine": "arm64", "metrics": {"expected_phase": 0.375, "max_probability_error": 3.543330795441335e-08, "mode": 3}, "mettleq_median_ms": 1.4072910125833005, "notebook": "pennylane/07_qft_and_qpe.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 1.270791981369257, "reference_over_mettleq": 0.9030058246705644, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}


## What should you conclude?

This shows a drop-in device change; meaningful speedups require a larger counting-plus-system register.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.